In [ ]:
# Import libraries and modules
import pandas as pd
import warnings
import pickle
import sys
import os
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
from pytorch_tabnet.tab_model import TabNetRegressor
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

print("[i] INFO - Program has been initialised")

[i] INFO - Program has been initialised


In [2]:
# Hide warnings from deep learning model in output
warnings.filterwarnings("ignore", module="pytorch_tabnet")

In [3]:
# Pre-define variables
rs = 1
folder, input = 'datasets', 'dataset_proc.csv'
ports = [53, 1883, 80, 443]

In [ ]:
def load_dataset():
    '''
    Load the dataset from the specified path.

    Parameters:
        None

    Returns:
        df
    '''

    path = os.path.join(folder, input)
    print(f"[i] INFO - Loading '{input}' dataset...")
    try:
        df = pd.read_csv(path, low_memory=False)
        print(f"[~] DEBUG - Loaded dataset with shape: ({df.shape[0]} x {df.shape[1]})")
    except FileNotFoundError:
        print(f"[!] ERROR - No file found at '{path}'")
        input();sys.exit(1)
    except PermissionError:
        print(f"[!] ERROR - No permission to read '{path}'")
        input();sys.exit(1)
    except UnicodeDecodeError:
        print(f"[!] ERROR - Unable to decode '{path}'")
        input();sys.exit(1)
    except pd.errors.EmptyDataError:
        print(f"[!] ERROR - No data found in '{path}'")
        input();sys.exit(1)
    except pd.errors.ParserError:
        print(f"[!] ERROR - Unable to parse data in '{path}'")
        input();sys.exit(1)

    return df

In [5]:
def prepare_dataframe(df, port):
    '''
    Prepare the dataframe by splitting into testing and training sets, encoding and standardizing the features.

    Parameters:
        df, port

    Returns:
        x_train, x_test, y_train, y_test 
    '''

    print(f"[i] INFO - Preparing '{input}' dataframe for port '{port}'...")

    # Split the dataframe into testing and training sets
    print("[i] INFO - Splitting the dataframe into testing and training sets...")

    x, y = df.drop(columns=['flow_duration']), df["flow_duration"]
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, random_state=rs)
    
    return x_train, x_test, y_train, y_test 

In [6]:
def calculate_score(x_test, y_test, model):
    '''
    Calculate and display the confusion matrix of the model.

    Parameters:
        x_test, y_test, model

    Returns:
        None
    '''

    print("[i] INFO - Calculating the scores of the model...")
    y_pred = model.predict(x_test)

    print(f"[i] INFO - R² ≈ '{r2_score(y_test, y_pred):.3f}' and MSE ≈ '{mean_squared_error(y_test, y_pred):.3f}'")

    return

In [7]:
def random_forest(x_train, x_test, y_train, y_test):
    '''
    Evaluate the random forest model using the processed dataset. 

    Parameters:
        x_train, x_test, y_train, y_test 

    Returns:
        None
    '''
    
    print("[i] INFO - Evaluating the random forest model...")
    
    # Perform random forest training
    model = RandomForestRegressor(n_estimators=300, max_depth=15, min_samples_split=5, random_state=rs, n_jobs=-1)
    model.fit(x_train, y_train)

    # Calculate the score of the model
    calculate_score(x_test, y_test, model)

    # Save model as file
    with open('randomforest.model', 'wb') as f:
        pickle.dump(model, f)
    
    return

In [8]:
def xg_boost(x_train, x_test, y_train, y_test):
    '''
    Evaluate the XGBoost model using the processed dataset. 

    Parameters:
        x_train, x_test, y_train, y_test 

    Returns:
        None
    '''
    
    print("[i] INFO - Evaluating the XGBoost model...")
    
    # Perform XGBoost training
    model = XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=rs, verbosity=0)
    model.fit(x_train, y_train)

    # Calculate the score of the model
    calculate_score(x_test, y_test, model)

    # Save model as file
    with open('xgboost.model', 'wb') as f:
        pickle.dump(model, f)
    
    return

In [9]:
def light_gbm(x_train, x_test, y_train, y_test):
    '''
    Evaluate the LightGBM model using the processed dataset. 

    Parameters:
        x_train, x_test, y_train, y_test 

    Returns:
        None
    '''
    
    print("[i] INFO - Evaluating the LightGBM model...")
    
    # Perform LightGBM training
    model = LGBMRegressor(n_estimators=300, num_leaves=31, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=rs, verbose=-1)
    model.fit(x_train, y_train)

    # Calculate the score of the model
    calculate_score(x_test, y_test, model)

    # Save model as file
    with open('lightgbm.model', 'wb') as f:
        pickle.dump(model, f)
    
    return

In [10]:
def tab_net(x_train, x_test, y_train, y_test):
    '''
    Evaluate the TabNet deep learning model using the processed dataset. 

    Parameters:
        x_train, x_test, y_train, y_test 

    Returns:
        None
    '''
    
    print("[i] INFO - Evaluating the TabNet deep learning model...")

    # Convert pd.df to np.array
    x_train, x_test = x_train.values, x_test.values
    y_train, y_test = y_train.values.reshape(-1, 1), y_test.values.reshape(-1, 1)
    
    # Perform TabNet training
    model = TabNetRegressor(seed=rs, verbose=0)
    model.fit(x_train, y_train, eval_set=[(x_test, y_test)], max_epochs=100, patience=100)

    # Calculate the score of the model
    calculate_score(x_test, y_test, model)

    # Save model as file
    with open('tabnet.model', 'wb') as f:
        pickle.dump(model, f)
    
    return

In [11]:
def main():
    '''
    Evaluate each machine learning model.

    Parameters:
        None

    Returns:
        None
    '''
    # Load the dataset
    df = load_dataset()

    for port in ports:
        # Select entries that match current port
        df_port = df[df["id.resp_p"] == port].copy()
        
        # Prepare the dataset
        x_train, x_test, y_train, y_test = prepare_dataframe(df_port, port)

        # Evaluate the random forest model
        random_forest(x_train, x_test, y_train, y_test)

        # Evaluate the XGBoost model
        xg_boost(x_train, x_test, y_train, y_test)

        # Evaluate the LightGBM model
        light_gbm(x_train, x_test, y_train, y_test)

        # Evaluate the TabNet deep learning model
        tab_net(x_train, x_test, y_train, y_test)
        
    print("[i] INFO - Program has finished execution")
    
if __name__ == "__main__":
    main()

[i] INFO - Loading 'dataset_proc.csv' dataset...
[~] DEBUG - Loaded dataset with shape: (16954 x 34)
[i] INFO - Preparing 'dataset_proc.csv' dataframe for port '53'...
[i] INFO - Splitting the dataframe into testing and training sets...
[i] INFO - Evaluating the random forest model...
[i] INFO - Calculating the scores of the model...
[i] INFO - R² ≈ '0.875' and MSE ≈ '0.015'
[i] INFO - Evaluating the XGBoost model...
[i] INFO - Calculating the scores of the model...
[i] INFO - R² ≈ '0.939' and MSE ≈ '0.007'
[i] INFO - Evaluating the LightGBM model...
[i] INFO - Calculating the scores of the model...
[i] INFO - R² ≈ '0.740' and MSE ≈ '0.031'
[i] INFO - Evaluating the TabNet deep learning model...
[i] INFO - Calculating the scores of the model...
[i] INFO - R² ≈ '0.829' and MSE ≈ '0.021'
[i] INFO - Preparing 'dataset_proc.csv' dataframe for port '1883'...
[i] INFO - Splitting the dataframe into testing and training sets...
[i] INFO - Evaluating the random forest model...
[i] INFO - Calcu